# MOSTA Lineage Sankey And 3D


In [ ]:
from pathlib import Path
import json
import shutil
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "assets").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate cytobridge-downstream repository root.")


DOWNSTREAM_ROOT = find_repo_root()
VENDOR_ROOT = DOWNSTREAM_ROOT / "vendor"
for path in (DOWNSTREAM_ROOT, VENDOR_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from downstream_helpers import (
    MostaRunConfig,
    build_mosta_interpolation_kwargs,
    display_html_outputs,
    display_svg_outputs,
    export_plotly_figure,
    list_output_files,
    load_mosta_context,
    resolve_mosta_output_dir,
)
from CytoBridge.tl import (
    compute_timepoint_communications,
    load_label_to_color,
    plot_lineage_sankey,
    plot_spatiotemporal_3d,
    run_interpolation_workflow,
)
from evaluation.arista_code.arista_helpers import plot_sankey
from evaluation.arista_code.arista_helpers_focus_anchor import plot_3d_spatial_sankey_style_focus_anchor


In [ ]:
config = MostaRunConfig(
    output_name="mosta_lineage_sankey_3d_notebook",
    piecewise_spatial_warp=False,
    skip_export=False,
    skip_snapshots=True,
    classifier_cache_path="assets/mosta/classifier_cache/classifier_resmlp_52fb7dc647bfe334.pt",
)
context = load_mosta_context()
output_dir = resolve_mosta_output_dir(config)
vector_scale = 1.0
png_scale = 2.0
output_dir


In [ ]:
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

interpolation = run_interpolation_workflow(
    **build_mosta_interpolation_kwargs(
        context=context,
        config=config,
        output_dir=output_dir,
    )
)

label_to_color = load_label_to_color(
    context.df[context.assets.annotation_key].astype(str).values,
    label_color_json=str(context.assets.label_color_json),
    color_h5ad=context.assets.color_h5ad,
    annotation_key=context.assets.annotation_key,
)
with open(output_dir / "label_to_color.json", "w", encoding="utf-8") as handle:
    json.dump(label_to_color, handle, indent=2)

fig_sankey = plot_lineage_sankey(
    plot_fn=plot_sankey,
    predicted_labels_list=interpolation.predicted_labels_list,
    time_keys=interpolation.time_keys,
    label_to_color=label_to_color,
    out_html=str(output_dir / "lineage_sankey.html"),
    min_flow=None,
    keep_source_cumfrac=0.8,
    normalize_mode=None,
    style="nature-methods",
    title="Cell Fate Transitions",
)

all_time_communications = compute_timepoint_communications(
    adata_dict=interpolation.adata_dict,
    time_points=interpolation.plot_3d_ts_points,
    annotation_key=context.assets.annotation_key,
    f_net=context.runtime.f_net,
    device=context.device,
    out_dir=str(output_dir / "attention"),
    save_dense_attention_matrix=False,
    remove_self_loop=False,
    winsor_quantile=0.995,
    save_pickle_path=str(output_dir / "mosta_all_time_communications.pkl"),
)

fig_3d = plot_spatiotemporal_3d(
    plot_fn=plot_3d_spatial_sankey_style_focus_anchor,
    adata_dict=interpolation.adata_dict,
    all_time_communications=all_time_communications,
    time_keys=interpolation.plot_3d_time_keys,
    plot_time_points=interpolation.plot_3d_ts_points,
    ts_points=interpolation.ts_points,
    observed_time_points=interpolation.observed_time_points,
    interp_points=interpolation.interp_points,
    annotation_key=context.assets.annotation_key,
    label_to_color=label_to_color,
    out_html=str(output_dir / "spatiotemporal_3d.html"),
    predicted_labels_list=interpolation.predicted_labels_list,
    spatial_key="spatial",
    z_spacing=1.0,
    reverse_time_order=True,
    intra_threshold=0.0,
    edge_focus_celltype="Brain",
    edge_top_k=5,
    edge_top_k_focus_label="Brain",
    ribbon_min_count=None,
    ribbon_keep_source_cumfrac=0.8,
    ribbon_focus_celltype="Brain",
    ribbon_focus_source_only=True,
    ribbon_focus_target_only=False,
    background_color=None,
    font_color="#1a1a1a",
    anchor_mode="centroid",
    anchor_subsample=1000,
    highlight_endpoints=True,
    endpoint_size=6,
    endpoint_opacity=0.9,
    edge_color="rgba(25,25,25,0.75)",
    edge_line_width_base=5,
    edge_line_width_scale=0.7,
    bidirectional_offset=0.2,
    bidirectional_curve=True,
    bidirectional_curve_points=18,
    ribbon_line_width_base=6,
    ribbon_line_width_scale=1.0,
    ribbon_line_alpha=0.55,
    ribbon_line_curve=0.12,
    ribbon_line_points=18,
    point_size=1.0,
    observed_point_subsample=None,
    generated_point_subsample=None,
    observed_point_alpha=0.7,
    generated_point_alpha=0.7,
    slices_only=False,
    show_time_axis=False,
    show_legend=False,
    show_title=False,
    show_slice_border=True,
    slice_border_width=5,
    slice_border_color_observed="#5f6a72",
    slice_border_color_generated="#8c6d5a",
    slice_fill_color_observed="#e6f0f6",
    slice_fill_color_generated="#f6eee5",
    slice_fill_opacity=0.5,
    width=1400,
    height=1000,
    focus_anchor_label="Brain",
    focus_anchor_k=None,
    focus_anchor_frac=0.2,
    focus_anchor_radius=None,
    focus_anchor_min_count=None,
)

export_plotly_figure(
    fig_sankey,
    svg_path=output_dir / "lineage_sankey.svg",
    pdf_path=output_dir / "lineage_sankey.pdf",
    vector_scale=vector_scale,
)
export_plotly_figure(
    fig_3d,
    svg_path=output_dir / "spatiotemporal_3d.svg",
    pdf_path=output_dir / "spatiotemporal_3d.pdf",
    png_path=output_dir / "spatiotemporal_3d.png",
    vector_scale=vector_scale,
    png_scale=png_scale,
)

output_dir


In [ ]:
list_output_files(output_dir)


In [ ]:
svg_paths = [
    output_dir / 'lineage_sankey.svg']
display_svg_outputs(svg_paths)

## Notes

- This notebook exports `lineage_sankey.svg/.pdf` and `spatiotemporal_3d.svg/.pdf/.png` alongside the interactive HTML outputs.
- If an iframe still does not render in your notebook frontend, use the link printed below each panel to open the HTML file in a new tab.
